# 02 — Classical baselines on the 28 test users (both EMG representations)

Reads the `A6_*` results JSONs produced by the laptop loop in `CURRENT_TODOs.md` (one folder per representation, `~/gest_results/A6_primary/<repr>/` (set `A6_RESULTS` to point at `A6_grid` for the full grid)).

The aggregator checks that every method used the committed test episodes (hash) and covers all 28 users. **These are reportable numbers.** They are reported for *both* representations; the representation choice comes from notebook 01 (dev users), not from this table.

Stats: user = unit, 95% bootstrap CI over users, paired Wilcoxon (Pratt) + Holm (`system/analysis/stats_utils.py`).

In [ ]:
import os, sys, pickle, json
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while not (REPO / ".git").exists():
    REPO = REPO.parent
os.chdir(REPO)
for rel in ("system", "system/MAML"):
    sys.path.insert(0, str(REPO / rel))
os.environ.setdefault("CODE_DIR", str(REPO))
os.environ.setdefault("RUN_DIR", os.environ.get("TEMP", "/tmp"))
os.environ.setdefault("HPO_CONFIG", "legacy")

from episodes import episode_cache as ec
from episodes.episode_numpy import materialize
from MAML.maml_data_pipeline import reorient_tensor_dict

DS = REPO / "dataset" / "meta-learning-sup-que-ds"
def load_dict(name, emg_ch):
    d = pickle.load(open(DS / name, "rb"))
    return d, reorient_tensor_dict(d, {"emg_in_ch": emg_ch, "imu_in_ch": 72})   # -> (trials, C, T)


In [ ]:
import pandas as pd
from analysis import aggregate as agg_mod
from analysis import stats_utils as su

RESULTS = Path(os.environ.get("A6_RESULTS", Path.home() / "gest_results" / "A6_primary"))
agg = {}
for rep_dir in sorted(p for p in RESULTS.iterdir() if p.is_dir()):
    files = sorted(rep_dir.glob("A6_*_results.json"))
    if not files:
        continue
    # aggregate one method at a time so a missing fold in one method doesn't block the others
    by_method = {}
    for f in files:
        by_method.setdefault(json.loads(f.read_text())["ablation_id"], []).append(f)
    for method, fs in sorted(by_method.items()):
        try:
            a = agg_mod.aggregate(fs)
        except ValueError as e:
            print(f"SKIP {rep_dir.name}/{method}: {e}")
            continue
        for m, cells in a.items():
            agg[f"{m} [{rep_dir.name}]"] = cells
print(len(agg), "method x representation entries")

In [ ]:
def table(cell):
    rows = agg_mod.summary_table(agg, cell)
    df = pd.DataFrame(rows).set_index("method")
    for c in ("mean", "ci_lo", "ci_hi", "sd_users"):
        df[c] = (df[c] * 100).round(1)
    return df.sort_values("mean", ascending=False)

print("PRIMARY CELL: 10-way 1-shot")
table((10, 1))

In [ ]:
# Full grid: mean accuracy (%) per method and cell
cells = sorted({c for v in agg.values() for c in v})
grid = pd.DataFrame({f"{n}w{k}s": {m: 100 * np.mean(list(v[(n, k)].values())) for m, v in agg.items() if (n, k) in v}
                     for n, k in cells})
grid.round(1).sort_values("10w1s", ascending=False) if "10w1s" in grid else grid.round(1)

In [ ]:
# Paired: fs64 vs env20 for each method (reported, not used for selection)
rows = []
for m in sorted({k.rsplit(" [", 1)[0] for k in agg}):
    a, b = f"{m} [fs64]", f"{m} [env20]"
    if a in agg and b in agg and (10, 1) in agg[a] and (10, 1) in agg[b]:
        users = sorted(agg[a][(10, 1)])
        r = su.paired_compare([agg[a][(10, 1)][u] for u in users], [agg[b][(10, 1)][u] for u in users])
        rows.append({"method": m, **{k: r[k] for k in ("mean_diff", "ci_lo", "ci_hi", "p", "wins", "losses")}})
if rows:
    df = pd.DataFrame(rows).set_index("method")
    df["p_holm"] = su.holm(df["p"].tolist())
    for c in ("mean_diff", "ci_lo", "ci_hi"):
        df[c] = (df[c] * 100).round(1)
    display(df)
else:
    print("need both env20 and fs64 results for at least one method")